<a href="https://colab.research.google.com/github/henrypitts/Baseball-Colab-Tools/blob/main/Arm_Angle_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

df["arm_angle"] = pd.to_numeric(
    df["arm_angle"],
    errors="coerce"
)
print(df.columns.tolist())

Saving stats7_13_26.csv to stats7_13_26 (1).csv
['last_name, first_name', 'player_id', 'year', 'p_formatted_ip', 'pa', 'k_percent', 'bb_percent', 'batting_avg', 'slg_percent', 'on_base_percent', 'on_base_plus_slg', 'p_era', 'xba', 'xslg', 'woba', 'xwoba', 'sweet_spot_percent', 'barrel_batted_rate', 'hard_hit_percent', 'avg_best_speed', 'avg_hyper_speed', 'whiff_percent', 'swing_percent', 'pitch_hand', 'arm_angle']


In [ ]:


bins = [ 10, 20, 30, 40, 50, 60, 70]

labels = [
    "10-19.9°",
    "20-29.9°",
    "30-39.9°",
    "40-49.9°",
    "50-59.9°",
    "60-69.9°"
]

df["arm_angle_group"] = pd.cut(
    df["arm_angle"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

grouped_pitchers = df[
    [
        "last_name, first_name",
        "player_id",
        "arm_angle",
        "pitch_hand",
        "arm_angle_group"
    ]
].sort_values(
    by=["arm_angle_group", "arm_angle"]
)

display(grouped_pitchers)

,"last_name, first_name",player_id,arm_angle,pitch_hand,arm_angle_group
95,"Sale, Chris",519242,10.6,L,10-19.9°
104,"Hancock, Emerson",676106,11.7,R,10-19.9°
125,"Lodolo, Nick",666157,14.5,L,10-19.9°
32,"Manaea, Sean",640455,15.2,L,10-19.9°
76,"Castillo, Luis",622491,16.3,R,10-19.9°
...,...,...,...,...,...
160,"Martinez, Nick",607259,55.1,R,50-59.9°
9,"Bibee, Tanner",676440,55.7,R,50-59.9°
85,"Bradley, Taj",671737,56.5,R,50-59.9°
163,"Pallante, Andre",669467,57.6,R,50-59.9°


In [ ]:
pitchers_by_arm_angle_and_era = df[[
    "last_name, first_name",
    "arm_angle_group",
    "pitch_hand",
    "arm_angle",
    "p_era"
]].dropna(subset=["p_era"]).sort_values(by=["arm_angle_group", "p_era"])

display(pitchers_by_arm_angle_and_era)

for group_name, group_df in pitchers_by_arm_angle_and_era.groupby('arm_angle_group', observed=True):
    print(f"### Arm Angle Group: {group_name}")
    display(group_df.drop(columns=['arm_angle_group']))


,"last_name, first_name",arm_angle_group,pitch_hand,arm_angle,p_era
95,"Sale, Chris",10-19.9°,L,10.6,2.20
104,"Hancock, Emerson",10-19.9°,R,11.7,3.23
153,"Lord, Brad",10-19.9°,R,17.1,3.88
15,"Arrighetti, Spencer",10-19.9°,R,18.9,4.50
32,"Manaea, Sean",10-19.9°,L,15.2,4.56
...,...,...,...,...,...
93,"Assad, Javier",50-59.9°,R,51.1,4.11
122,"Cameron, Noah",50-59.9°,L,51.8,4.89
1,"Nelson, Ryne",50-59.9°,R,54.6,4.97
12,"Avila, Luinder",50-59.9°,R,51.7,5.08


### Arm Angle Group: 10-19.9°


,"last_name, first_name",pitch_hand,arm_angle,p_era
95,"Sale, Chris",L,10.6,2.20
104,"Hancock, Emerson",R,11.7,3.23
153,"Lord, Brad",R,17.1,3.88
15,"Arrighetti, Spencer",R,18.9,4.50
32,"Manaea, Sean",L,15.2,4.56
125,"Lodolo, Nick",L,14.5,4.60
76,"Castillo, Luis",R,16.3,4.93
34,"Nola, Aaron",R,19.7,5.75


### Arm Angle Group: 20-29.9°


,"last_name, first_name",pitch_hand,arm_angle,p_era
17,"Misiorowski, Jacob",R,29.8,1.62
41,"Wheeler, Zack",R,22.2,2.28
0,"Newcomb, Sean",L,25.8,2.52
80,"Ryan, Joe",R,23.7,2.85
54,"Soroka, Michael",R,26.8,3.07
33,"Tolle, Payton",L,27.2,3.14
148,"Scott, Christian",R,29.9,3.17
107,"Ramírez, Yohan",R,25.7,3.31
24,"King, Michael",R,24.4,3.41
119,"Skenes, Paul",R,25.6,3.58


### Arm Angle Group: 30-39.9°


... (3 outputs left out)

### Arm Angle Group: 50-59.9°


,"last_name, first_name",pitch_hand,arm_angle,p_era
139,"Burns, Chase",R,52.8,2.54
65,"Cease, Dylan",R,53.0,2.56
160,"Martinez, Nick",R,55.1,2.65
102,"Rasmussen, Drew",R,52.5,3.26
161,"Ashby, Aaron",L,52.5,3.56
66,"Bradish, Kyle",R,53.0,3.61
70,"Cantillo, Joey",L,52.0,3.66
85,"Bradley, Taj",R,56.5,3.67
124,"Wacha, Michael",R,53.1,3.77
9,"Bibee, Tanner",R,55.7,3.90


### Arm Angle Group: 60-69.9°


,"last_name, first_name",pitch_hand,arm_angle,p_era
132,"Yesavage, Trey",R,65.8,3.72


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import f_oneway, kruskal
from statsmodels.stats.multitest import multipletests

# Define performance columns
performance_columns = [
    "p_era",
    "k_percent",
    "bb_percent",
    "woba",
    "xwoba",
    "whiff_percent",
    "hard_hit_percent",
    "barrel_batted_rate",
    "sweet_spot_percent",
    "swing_percent"
]

# Ensure columns are numeric
for column in performance_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# 1. Calculate Average Performance by Arm Angle Group
average_performance = (
    df.groupby("arm_angle_group", observed=False)[performance_columns]
      .mean()
      .reset_index()
)

pitcher_counts = (
    df.groupby("arm_angle_group", observed=False)
      .size()
      .reset_index(name="pitcher_count")
)

average_performance = pitcher_counts.merge(
    average_performance,
    on="arm_angle_group"
)

# Add average K-BB%
average_performance["k_bb_percent"] = (
    average_performance["k_percent"]
    - average_performance["bb_percent"]
)

average_performance = average_performance.round({
    "p_era": 2,
    "k_percent": 1,
    "bb_percent": 1,
    "k_bb_percent": 1,
    "woba": 3,
    "xwoba": 3,
    "whiff_percent": 1,
    "hard_hit_percent": 1,
    "barrel_batted_rate": 1,
    "sweet_spot_percent": 1,
    "swing_percent": 1
})

print("### Average Performance Metrics by Arm Angle Group:")
display(average_performance)

# 2. Calculate 95% Confidence Intervals
results_ci = []

for group_name, group in df.groupby(
    "arm_angle_group",
    observed=True,
    sort=False
):
    for column in performance_columns:
        values = group[column].dropna()
        n = len(values)

        if n >= 2:
            mean = values.mean()
            std = values.std(ddof=1) if n > 1 else np.nan

            if pd.notna(std) and std > 0:
                standard_error = std / np.sqrt(n)
                critical_value = stats.t.ppf(
                    0.975,
                    df=n - 1
                )
                margin = critical_value * standard_error
            else:
                # If std is 0 or nan, CI is just the mean
                margin = 0

            results_ci.append({
                "arm_angle_group": str(group_name),
                "metric": column,
                "n": n,
                "mean": mean,
                "std_dev": std,
                "ci_95_low": mean - margin,
                "ci_95_high": mean + margin
            })

confidence_intervals = pd.DataFrame(results_ci)

confidence_intervals = confidence_intervals.round({
    "mean": 3,
    "std_dev": 3,
    "ci_95_low": 3,
    "ci_95_high": 3
})

print("\n### 95% Confidence Intervals for Performance Metrics:")
display(confidence_intervals)

# 3. Perform ANOVA and Kruskal-Wallis Tests
results_tests = []

for metric in performance_columns:
    samples = []
    for _, group in df.groupby("arm_angle_group", observed=True):
        values = group[metric].dropna()
        if len(values) >= 2:
            samples.append(values)

    if len(samples) >= 2:
        anova_stat, anova_p = f_oneway(*samples)
        kruskal_stat, kruskal_p = kruskal(*samples)

        results_tests.append({
            "metric": metric,
            "anova_p": anova_p,
            "kruskal_p": kruskal_p
        })

test_results = pd.DataFrame(results_tests)

# Correct for multiple testing
test_results["anova_fdr_p"] = multipletests(
    test_results["anova_p"],
    method="fdr_bh"
)[1]

test_results["kruskal_fdr_p"] = multipletests(
    test_results["kruskal_p"],
    method="fdr_bh"
)[1]

test_results["anova_significant"] = (
    test_results["anova_fdr_p"] < 0.05
)

test_results["kruskal_significant"] = (
    test_results["kruskal_fdr_p"] < 0.05
)

print("\n### Statistical Test Results (ANOVA and Kruskal-Wallis) for Variation Between Groups:")
display(
    test_results.sort_values("anova_p").round(4)
)


### Average Performance Metrics by Arm Angle Group:


,arm_angle_group,pitcher_count,p_era,k_percent,bb_percent,woba,xwoba,whiff_percent,hard_hit_percent,barrel_batted_rate,sweet_spot_percent,swing_percent,k_bb_percent
0,10-19.9°,8,4.21,22.8,8.1,0.318,0.326,24.0,38.7,8.2,34.3,48.5,14.6
1,20-29.9°,30,4.28,22.6,8.4,0.314,0.318,24.4,38.3,7.8,34.5,47.4,14.2
2,30-39.9°,56,4.26,21.8,8.0,0.318,0.322,24.5,40.5,8.4,34.6,48.1,13.7
3,40-49.9°,54,4.14,21.5,8.1,0.313,0.317,24.1,38.9,8.2,33.6,47.8,13.4
4,50-59.9°,16,3.76,22.4,8.6,0.308,0.312,24.7,38.4,7.7,33.4,46.6,13.8
5,60-69.9°,1,3.72,22.2,12.5,0.280,0.284,29.3,35.3,5.5,36.3,47.0,9.7



### 95% Confidence Intervals for Performance Metrics:


,arm_angle_group,metric,n,mean,std_dev,ci_95_low,ci_95_high
0,20-29.9°,p_era,30,4.285,1.281,3.807,4.763
1,20-29.9°,k_percent,30,22.577,5.557,20.502,24.652
2,20-29.9°,bb_percent,30,8.393,2.561,7.437,9.350
3,20-29.9°,woba,30,0.314,0.042,0.298,0.330
4,20-29.9°,xwoba,30,0.318,0.038,0.304,0.332
5,20-29.9°,whiff_percent,30,24.373,3.999,22.880,25.867
6,20-29.9°,hard_hit_percent,30,38.310,4.963,36.457,40.163
7,20-29.9°,barrel_batted_rate,30,7.797,2.785,6.757,8.837
8,20-29.9°,sweet_spot_percent,30,34.523,3.472,33.227,35.820
9,20-29.9°,swing_percent,30,47.353,3.264,46.135,48.572



### Statistical Test Results (ANOVA and Kruskal-Wallis) for Variation Between Groups:


,metric,anova_p,kruskal_p,anova_fdr_p,kruskal_fdr_p,anova_significant,kruskal_significant
6,hard_hit_percent,0.1428,0.1391,0.9817,0.9958,False,False
9,swing_percent,0.2685,0.2386,0.9817,0.9958,False,False
8,sweet_spot_percent,0.4602,0.3818,0.9817,0.9958,False,False
0,p_era,0.6524,0.6187,0.9817,0.9958,False,False
7,barrel_batted_rate,0.7155,0.7827,0.9817,0.9958,False,False
4,xwoba,0.7320,0.7491,0.9817,0.9958,False,False
1,k_percent,0.8206,0.9267,0.9817,0.9958,False,False
2,bb_percent,0.8591,0.9641,0.9817,0.9958,False,False
3,woba,0.8892,0.8098,0.9817,0.9958,False,False
5,whiff_percent,0.9817,0.9958,0.9817,0.9958,False,False


In [ ]:
pitcher_name = input("Enter the full name of the pitcher (e.g., 'Verlander, Justin'): ")
target_pitcher = df[df['last_name, first_name'] == pitcher_name]

if not target_pitcher.empty:
    target_arm_angle = target_pitcher['arm_angle'].iloc[0]

    # Define the 1-degree range
    lower_bound = target_arm_angle - 2
    upper_bound = target_arm_angle + 2

    # Filter pitchers within this range
    pitchers_in_range = df[
        (df['arm_angle'] >= lower_bound) &
        (df['arm_angle'] <= upper_bound)
    ].sort_values(by='arm_angle')

    print(f"\nPitchers with arm angles within 2 degree of {pitcher_name} (Arm Angle: {target_arm_angle:.1f}°):\n")
    display(pitchers_in_range[['last_name, first_name', 'pitch_hand', 'arm_angle',"p_era"]])
else:
    print(f"Pitcher '{pitcher_name}' not found in the dataset. Please check the name and try again.")

Enter the full name of the pitcher (e.g., 'Verlander, Justin'): Elder, Bryce

Pitchers with arm angles within 2 degree of Elder, Bryce (Arm Angle: 48.3°):



,"last_name, first_name",pitch_hand,arm_angle,p_era
83,"Paddack, Chris",R,46.3,6.79
44,"Leiter, Jack",R,46.3,5.29
112,"Painter, Andrew",R,46.4,7.06
51,"Liberatore, Matthew",L,46.6,5.00
21,"Skubal, Tarik",L,46.9,3.06
133,"Springs, Jeffrey",L,47.0,6.08
105,"Woods Richardson, Simeon",R,47.1,6.40
52,"Gray, Sonny",R,47.2,2.54
10,"Junk, Janson",R,47.4,4.57
114,"Gore, MacKenzie",L,47.8,4.72


In [ ]:
!pip -q install nbconvert

from pathlib import Path
from google.colab import files

# Find notebooks in Colab
notebooks = list(Path("/content").glob("*.ipynb"))

print("Available notebooks:")
for nb in notebooks:
    print(nb)

Available notebooks:
/content/Arm_Angle_Search.ipynb


In [ ]:
from google.colab import files

uploaded = files.upload()
notebook_path = next(iter(uploaded))

print(f"Notebook found: {notebook_path}")

Saving Arm_Angle_Search.ipynb to Arm_Angle_Search.ipynb
Notebook found: Arm_Angle_Search.ipynb


In [ ]:
from pathlib import Path
from IPython.display import display
from google.colab import files

# Select your notebook
notebook_path = Path("/content/Arm_Angle_Search.ipynb")

# Export to HTML
!jupyter nbconvert --to html "{notebook_path}" \
    --output "Notebook_Report.html" \
    --output-dir "/content"

# Download HTML
files.download("/content/Notebook_Report.html")

[NbConvertApp] Converting notebook /content/Arm_Angle_Search.ipynb to html
[NbConvertApp] Writing 416475 bytes to /content/Notebook_Report.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>